# 👥 Phase 3: Customer Behavior & RFM Segmentation
**Project:** E-Commerce Sales Analytics Portfolio Project
**Objective:** Segment customer base into actionable behavioral cohorts (Champions, Loyal Customers, Potential Loyalists, At Risk, Lost) using Recency, Frequency, and Monetary (RFM) modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

df = pd.read_csv(os.path.join('..', 'data', 'cleaned', 'superstore_cleaned.csv'))
df['order_date'] = pd.to_datetime(df['order_date'])
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)
print(f'Snapshot Date: {snapshot_date.date()}')

## 1. Calculating RFM Metrics per Customer
- **Recency (R):** Number of days since most recent purchase
- **Frequency (F):** Total number of distinct orders placed
- **Monetary (M):** Total lifetime revenue generated

In [ ]:
rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'sales': 'sum',
    'profit': 'sum',
    'customer_name': 'first',
    'segment': 'first'
}).reset_index()

rfm.rename(columns={
    'order_date': 'recency_days',
    'order_id': 'frequency_orders',
    'sales': 'monetary_value',
    'profit': 'total_profit'
}, inplace=True)
rfm.head(5)

## 2. RFM Quantile Scoring & Behavioral Cohort Assignment

In [ ]:
rfm['r_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency_orders'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary_value'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def assign_rfm_segment(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 3:
        return 'Potential Loyalists'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m >= 3:
        return 'About to Sleep'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Hibernating / Lost'
    else:
        return 'Promising / Needs Attention'

rfm['customer_segment'] = rfm.apply(assign_rfm_segment, axis=1)
rfm.to_csv(os.path.join('..', 'data', 'cleaned', 'customer_rfm_segments.csv'), index=False)
print('Customer RFM Segmentation complete.')